In [1]:
from censored_regressors.metrics.evaluators import evaluate_observed, evaluate_latent
from censored_regressors.models.models_gpy import GP, TruncGP, CensoredGP_EP, CensoredGP_Laplace
from censored_regressors.models.models_gpytorch import GPyTorchOptimizer, CensoredGP_VI_gpytorch

import pickle

In [2]:
from censored_regressors.utils.tasks_dataloader import Housing

In [3]:
import numpy as np
import time

In [4]:
dataset = Housing()
dataset.load_data()

True

In [5]:
import pandas as pd
pd.Series(dataset.Y_obs.flatten()).describe()

count    506.000000
mean      22.532806
std        9.197104
min        5.000000
25%       17.025000
50%       21.200000
75%       25.000000
max       50.000000
dtype: float64

In [6]:
pd.Series(dataset.Y_obs.flatten()).value_counts()

50.0    16
25.0     8
22.0     7
21.7     7
23.1     7
        ..
32.9     1
34.6     1
30.3     1
33.3     1
8.1      1
Name: count, Length: 229, dtype: int64

In [7]:
task_mapping = {
    'Housing_50': Housing(upper_bound=50.0, is_natural=False),
    'Housing_25': Housing(upper_bound=25.0, is_natural=False),
    # Add other tasks here...
}

config = {
    'evaluations': [evaluate_observed, evaluate_latent],
    'methods': [GP, TruncGP, CensoredGP_EP, CensoredGP_Laplace, CensoredGP_VI_gpytorch],
    'kernels': ['rbf'],#, 'lin', 'matern32', 'matern52', 'lin_rbf', 'lin_matern32', 'lin_matern52'],
    'tasks': ['Housing_50'],
    'n_folds': 2,
}

results = {}

In [8]:
# # 1. Specify your filename
# filename = 'results_checkpoint.pkl'
# 
# # 2. Open the file in 'read binary' mode and load it
# with open(filename, 'rb') as f:
#     results = pickle.load(f)
# 
# # Now you can use it just like a normal dictionary
# print(my_dictionary.keys())

In [ ]:
import time
import pickle

# --- TRAINING LOOP ---

print(f"Starting experiment: {len(config['tasks'])} tasks x {len(config['methods'])} methods x {len(config['kernels'])} kernels.")

for task_name in config['tasks']:
    print(f"\n[Task: {task_name}]")
    
    if task_name not in task_mapping: continue
    dataset = task_mapping[task_name]
    if not dataset.load_data(): continue

    folds = dataset.get_cv_folds(n_folds=config['n_folds'], shuffle=True, seed=42)
    
    if task_name not in results: results[task_name] = {}

    for method_cls in config['methods']:
        base_name = getattr(method_cls, 'base_name', method_cls.__name__)
        is_vi = "VI" in base_name or "Variational" in base_name

        for kernel_ in config['kernels']:
            method_full_name = f"{base_name}_{kernel_}"
            
            if method_full_name not in results[task_name]: 
                results[task_name][method_full_name] = []
            
            if len(results[task_name][method_full_name]) == config['n_folds']:
                 print(f"  {method_full_name} (Skipping - Done)")
                 continue

            print(f"  Processing {method_full_name} ", end='', flush=True)
            fold_results = []
            
            for fold_i, (train_data, test_data) in enumerate(folds):
                # Unpack: 
                # train_data typically: (X, y_observed, censoring_indicator)
                X_train, Y_train_obs, C_train = train_data[:3]
                X_test, Y_test_obs, C_test = test_data[:3]
                
                # The latent truth (uncensored) is needed for evaluate_latent
                Y_test_true_hidden = test_data[3] if len(test_data) > 3 else None

                instance_name = f"{method_full_name}_fold{fold_i}"
                model = method_cls(kernel_type=kernel_, name=instance_name)
                
                t_st = time.time()
                try:
                    # Fit
                    # --- APPLY VI TUNING ---
                    if is_vi:
                        # We pass custom fit parameters for the VI model
                        # num_restarts helps find a better local optima for the ELBO
                        model.fit(
                            (X_train, Y_train_obs, C_train), 
                            optimizer='ngd',    # Natural Gradient Descent is often more stable
                            num_restarts=5,     # Increased restarts for VI
                            max_iters=3000,     # VI often needs more steps to converge
                            init_params='laplace' # Using Laplace init as it's usually the best starting point
                        )
                    else:
                        model.fit((X_train, Y_train_obs, C_train))
                    
                    # Predict
                    pred = model.predict(X_test)
                    t_pd = time.time() - t_st

                    # --- FIXED EVALUATION CALL ---
                    eval_func = config['evaluations'][0]
                    
                    # Logic to handle both evaluate_observed and evaluate_latent dynamically
                    eval_kwargs = {
                        'X': X_test,
                        'ground_truth_cens': Y_test_obs,
                        'pred': pred['f_mean'],
                        'gp_lower': pred['f_025'],
                        'gp_upper': pred['f_975'],
                        'censoring': C_test,
                        'model': model,
                        'likelihood': getattr(model, 'likelihood', None)
                    }

                    # Add latent truth only if using evaluate_latent
                    if eval_func.__name__ == 'evaluate_latent':
                        eval_kwargs['ground_truth_latent'] = Y_test_true_hidden

                    metric_scores = eval_func(**eval_kwargs)
                    
                    metric_scores['time'] = t_pd
                    fold_results.append(metric_scores)
                    print('.', end='', flush=True)
                
                except Exception as e:
                    print(f"X({fold_i})", end='', flush=True)
                    fold_results.append({'error': str(e)})
                

            results[task_name][method_full_name] = fold_results
            
            # Checkpoint
            with open('results_checkpoint.pkl', 'wb') as f:
                pickle.dump(results, f)
            
            print(" Done.")

print("\n\nExperiment Complete.")

Starting experiment: 1 tasks x 5 methods x 1 kernels.

[Task: Housing_50]
  Processing GP_Gauss_rbf 

reconstraining parameters GP_regression.rbf.lengthscale
reconstraining parameters GP_regression.Gaussian_noise.variance


.

reconstraining parameters GP_regression.rbf.lengthscale
reconstraining parameters GP_regression.Gaussian_noise.variance


. Done.
  Processing TruncGP_rbf 

...

reconstraining parameters gp_censored_regression.rbf.lengthscale
reconstraining parameters gp_censored_regression.CensoredGaussian.variance


Warning - optimization restart 2/10 failed
Warning - optimization restart 4/10 failed
Warning - optimization restart 10/10 failed
.

reconstraining parameters gp_censored_regression.rbf.lengthscale
reconstraining parameters gp_censored_regression.CensoredGaussian.variance


Warning - optimization restart 3/10 failed
Warning - optimization restart 5/10 failed
Warning - optimization restart 7/10 failed
Warning - optimization restart 8/10 failed
. Done.
  Processing CensoredGP_VI_gpytorch_rbf Fitting CensoredGP_VI_gpytorch_rbf_fold0 with NGD (Restarts: 5, Init: laplace)...
  [Restart 1] Attempting Native PyTorch Laplace Initialization...
  [+] Native Laplace initialization successful.


Loss: 10.335:  92%|█████████▏| 2747/3000 [08:22<00:06, 38.36it/s]  

In [16]:
import pandas as pd

def results_to_dataframe(results):
    rows = []

    # Level 1: Task (e.g., 'Housing_50')
    for task_name, method_dict in results.items():
        
        # Level 2: Method_Kernel (e.g., 'CensoredGP_VI_rbf')
        # Note: Your loop stores fold_results directly under method_full_name
        for method_full_name, fold_list in method_dict.items():
            
            # Level 3: Folds (List of dictionaries)
            for fold_id, metrics in enumerate(fold_list): 
                
                # Safety check: skip if this fold had an error
                if 'error' in metrics:
                    continue
                
                # 1. Split the method name and kernel for the dataframe
                # Assumes the format "MethodName_Kernel" (e.g., CensoredGP_EP_rbf)
                parts = method_full_name.rsplit('_', 1)
                method_base = parts[0]
                kernel_name = parts[1] if len(parts) > 1 else 'unknown'

                # 2. Create a base row with identifiers
                row = {
                    'Task': task_name,
                    'Method': method_base,
                    'Kernel': kernel_name,
                    'Fold': fold_id
                }
                
                # 3. Add all metrics to the row
                row.update(metrics)
                
                # 4. Standardize duration naming
                if 'time' in row:
                    row['Duration'] = row.pop('time')
                
                rows.append(row)

    if not rows:
        return pd.DataFrame()

    # 5. Create DataFrame
    df = pd.DataFrame(rows)

    # 6. Reorder columns for better readability
    id_cols = ['Task', 'Method', 'Kernel', 'Fold']
    
    # Updated to match the exact keys from your evaluation module
    preferred_metrics = [
        'NLPD', 'NLPD_latent', 'RMSE', 'MAE', 'MAE_c', 'MAE_nc', 
        'Hinge_MAE', 'Coverage', 'Accuracy', 'Precision', 'Recall'
    ]
    
    # Filter preferred metrics to only those that actually exist in the df
    existing_metrics = [c for c in preferred_metrics if c in df.columns]
    
    # Identify any extra columns (like 'Jaccard' or unknown metrics)
    extra_cols = [c for c in df.columns if c not in id_cols + existing_metrics + ['Duration']]
    
    # Final ordering (Duration at the end)
    final_cols = id_cols + existing_metrics + extra_cols + (['Duration'] if 'Duration' in df.columns else [])
    
    return df[final_cols]

In [17]:
# CORRECT USAGE: Pass the full dictionary
df_results = results_to_dataframe(results)

In [18]:
df_results

,Task,Method,Kernel,Fold,NLPD,RMSE,MAE,MAE_c,MAE_nc,Hinge_MAE,Coverage,Accuracy,Precision,Recall,Jaccard,Duration
0,Housing_50,GP_Gauss,rbf,0,2.475533,3.223049,2.178950,4.647163,2.108716,2.132105,0.719368,0.988142,1.0,0.571429,0.571429,15.770948
1,Housing_50,GP_Gauss,rbf,1,2.507509,3.317010,2.473232,2.772248,2.462202,2.465631,0.766798,0.972332,1.0,0.222222,0.222222,9.743954
2,Housing_50,TruncGP,rbf,0,2.809298,3.793988,2.272921,12.708149,1.975984,2.272921,0.620553,0.972332,0.0,0.000000,0.000000,5.784138
3,Housing_50,TruncGP,rbf,1,2.800324,3.915905,2.347499,7.902066,2.142617,2.342323,0.667984,0.976285,1.0,0.333333,0.333333,10.470383
4,Housing_50,CensoredGP_EP,rbf,0,3.272502,3.844387,2.623723,10.210745,2.407832,2.623723,0.897233,0.972332,0.0,0.000000,0.000000,4.158204
5,Housing_50,CensoredGP_EP,rbf,1,3.251584,3.456361,2.504693,8.819193,2.271781,2.504693,0.905138,0.964427,0.0,0.000000,0.000000,3.259316
6,Housing_50,CensoredGP_Laplace,rbf,0,2.416297,3.465334,2.283910,6.153492,2.173800,2.179116,0.703557,0.988142,1.0,0.571429,0.571429,13.829352
7,Housing_50,CensoredGP_Laplace,rbf,1,2.401854,3.179825,2.328537,1.964717,2.341957,2.289510,0.810277,0.984190,1.0,0.555556,0.555556,16.621062
8,Housing_50,CensoredGP_VI_gpytorch,rbf,0,0.089791,6.315193,4.246818,12.967832,3.998659,4.246818,0.498024,0.972332,0.0,0.000000,0.000000,2361.441905
9,Housing_50,CensoredGP_VI_gpytorch,rbf,1,0.101954,6.809812,4.526222,13.316466,4.201992,4.526222,0.525692,0.964427,0.0,0.000000,0.000000,359.285237


In [19]:
# Calculate summary
summary = df_results.groupby(['Task', 'Method', 'Kernel'])[['NLPD', 'MAE', 'Hinge_MAE', 'Duration']].mean()
summary

NLPD       MAE  Hinge_MAE  \
Task       Method                 Kernel                                  
Housing_50 CensoredGP_EP          rbf     3.262043  2.564208   2.564208   
           CensoredGP_Laplace     rbf     2.409076  2.306224   2.234313   
           CensoredGP_VI_gpytorch rbf     0.095873  4.386520   4.386520   
           GP_Gauss               rbf     2.491521  2.326091   2.298868   
           TruncGP                rbf     2.804811  2.310210   2.307622   

                                             Duration  
Task       Method                 Kernel               
Housing_50 CensoredGP_EP          rbf        3.708760  
           CensoredGP_Laplace     rbf       15.225207  
           CensoredGP_VI_gpytorch rbf     1360.363571  
           GP_Gauss               rbf       12.757451  
           TruncGP                rbf        8.127260

In [20]:
for method, folds in results['Housing_50'].items():
    if "VI" in method:
        errors = [f['error'] for f in folds if 'error' in f]
        print(f"{method} has {len(errors)} errors: {errors}")

CensoredGP_VI_gpytorch_rbf has 0 errors: []
